# Context-FID Score Presentation
## Necessary packages and functions call

- Context-FID score: A useful metric measures how well the the synthetic time series windows ”fit” into the local context of the time series

In [1]:
import os
import torch
import numpy as np
import sys
sys.path.append(os.path.join(os.path.dirname('__file__'), '../'))
from Utils.context_fid import Context_FID
from Utils.metric_utils import display_scores
from Utils.cross_correlation import CrossCorrelLoss

## Data Loading

Load original dataset and preprocess the loaded data.

In [2]:
iterations = 5
#ori_data = np.load('../toy_exp/samples/sine_ground_truth_24_train.npy')
#fake_data = np.load('../toy_exp/ddpm_fake_sines.npy')

#UNCONDITIONAL GENERATION
'''
ori_data = np.load('../OUTPUT/etth/samples/etth_norm_truth_24_train.npy') #Need to use normalized samples!
fake_data = np.load('../OUTPUT/etth/ddpm_fake_etth.npy')
'''

#PREDICTION

ori_data = np.load('../OUTPUT/sines/samples/sine_ground_truth_24_test.npy')
fake_data = np.load('../OUTPUT/sines/ddpm_predict-counterfactual_sines_l1_l2.npy')

## Context-FID Score

- The Frechet Inception distance-like score is based on unsupervised time series embeddings. It is able to score the fit of the fixed length synthetic samples into their context of (often much longer) true time series.

- The lowest scoring models correspond to the best performing models in downstream tasks

In [3]:
context_fid_score = []

for i in range(iterations):
    context_fid = Context_FID(ori_data[:], fake_data[:ori_data.shape[0]])
    context_fid_score.append(context_fid)
    print(f'Iter {i}: ', 'context-fid =', context_fid, '\n')
      
display_scores(context_fid_score)

Iter 0:  context-fid = 0.046280722452948606 

Iter 1:  context-fid = 0.029223037754435174 

Iter 2:  context-fid = 0.03770922799103803 

Iter 3:  context-fid = 0.03678288626259617 

Iter 4:  context-fid = 0.03832965712491396 

Final Score:  0.037665106317186385 ± 0.007519996563466639


## Correlational Score

- The metric uses the absolute error of the auto-correlation estimator by real data and synthetic data as the metric to assess the temporal dependency.

- For d > 1, it uses the l1-norm of the difference between cross correlation matrices.

In [80]:
def random_choice(size, num_select=100):
    select_idx = np.random.randint(low=0, high=size, size=(num_select,))
    return select_idx

In [81]:
x_real = torch.from_numpy(ori_data)
x_fake = torch.from_numpy(fake_data)

correlational_score = []
size = int(x_real.shape[0] / iterations)

for i in range(iterations):
    real_idx = random_choice(x_real.shape[0], size)
    fake_idx = random_choice(x_fake.shape[0], size)
    corr = CrossCorrelLoss(x_real[real_idx, :, :], name='CrossCorrelLoss')
    loss = corr.compute(x_fake[fake_idx, :, :])
    correlational_score.append(loss.item())
    print(f'Iter {i}: ', 'cross-correlation =', loss.item(), '\n')

display_scores(correlational_score)

Iter 0:  cross-correlation = 0.05509208200204531 

Iter 1:  cross-correlation = 0.05125659341799414 

Iter 2:  cross-correlation = 0.06890133070401747 

Iter 3:  cross-correlation = 0.06718241493069302 

Iter 4:  cross-correlation = 0.07489069841344183 

Final Score:  0.06346462389363836 ± 0.012308617091314495
